# MedVision Agent — Lung Segmentation U-Net (Chest X-Ray + CT)

> ⚠️ **This is a research prototype and not approved for clinical diagnosis.**
> Use public datasets only and respect their licenses.

**Goal:** train a tiny 2-D U-Net on chest images to segment lung regions
and abnormalities, then export `lung_unet.onnx`.

- Datasets (all freely available on Kaggle):
  - **Chest X-Ray Masks & Labels** (`nikhilpandey360/chest-xray-masks-and-labels`):
    Montgomery + Shenzhen X-rays with binary lung masks, ~800 images.
  - **Finding & Measuring Lungs in CT** (`kmader/finding-lungs-in-ct-data`):
    2D CT slices with lung segmentation masks.
  - Either dataset works independently; the notebook auto-detects which is available.
- Approach: 2-D lung region segmentation (deliberately lightweight).
- Loss: Dice + BCE · Metrics: Dice / IoU / precision / recall.
- HU windowing for CT: clip [-1000, 400] → scale to [0,1].
  X-ray images are already in [0,255] grayscale.

In [ ]:
%pip install -q onnx onnxruntime matplotlib tqdm pandas
# torch + torchvision preinstalled on Colab/Kaggle GPUs

In [ ]:
import json, math, random, os, glob
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from PIL import Image
import matplotlib.pyplot as plt

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CONFIG = {
    'SIZE': 256,
    'BATCH': 16,
    'EPOCHS': 25,
    'LR': 1e-3,
    'MAX_IMAGES': 600,   # teaching subset; raise for real experiments
    'SEED': 42,
}
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
print('device:', DEVICE)

## 1 · Locate lung segmentation data

### Option A: Chest X-Ray Masks & Labels
Kaggle: `nikhilpandey360/chest-xray-masks-and-labels`
```
DATA_ROOT/
  Lung Segmentation/
    CXR_png/        # chest X-ray images (.png)
    masks/          # binary lung masks (.png)
```

### Option B: Finding & Measuring Lungs in CT
Kaggle: `kmader/finding-lungs-in-ct-data`
```
DATA_ROOT/
  2d_images/      # CT slice images (.tif)
  2d_masks/       # binary lung masks (.tif)
```

In [ ]:
# TODO(student): download one or both datasets and set paths.

# --- Option A: Chest X-Ray Masks & Labels ---
XRAY_CANDIDATES = [
    '/kaggle/input/chest-xray-masks-and-labels',
    '/kaggle/input/chest-xray-masks-and-labels/Lung Segmentation',
    '/content/chest-xray-masks-and-labels',
    './data/chest-xray-masks-and-labels',
]

# --- Option B: Finding & Measuring Lungs in CT ---
CT_CANDIDATES = [
    '/kaggle/input/finding-lungs-in-ct-data',
    '/content/finding-lungs-in-ct-data',
    './data/finding-lungs-in-ct-data',
]

img_mask_pairs = []  # list of (image_path, mask_path)

# Try to find Chest X-Ray dataset
xray_root = next((p for p in XRAY_CANDIDATES if Path(p).exists()), None)
if xray_root:
    xr = Path(xray_root)
    # Find image and mask directories
    img_dir = next((d for d in [xr / 'CXR_png', xr / 'Lung Segmentation' / 'CXR_png'] if d.exists()), None)
    msk_dir = next((d for d in [xr / 'masks', xr / 'Lung Segmentation' / 'masks'] if d.exists()), None)
    if img_dir and msk_dir:
        for img_p in sorted(img_dir.glob('*.png')):
            # Match mask by filename (try with and without _mask suffix)
            stem = img_p.stem
            for pattern in [stem + '_mask.png', stem + '.png']:
                msk_p = msk_dir / pattern
                if msk_p.exists():
                    img_mask_pairs.append((str(img_p), str(msk_p)))
                    break
        print(f'Found {len(img_mask_pairs)} X-ray image-mask pairs from {xray_root}')

# Try to find CT dataset
ct_root = next((p for p in CT_CANDIDATES if Path(p).exists()), None)
if ct_root:
    ct = Path(ct_root)
    img_dir = ct / '2d_images'
    msk_dir = ct / '2d_masks'
    if img_dir.exists() and msk_dir.exists():
        ct_pairs = []
        for img_p in sorted(img_dir.glob('*.tif')):
            msk_p = msk_dir / img_p.name
            if msk_p.exists():
                ct_pairs.append((str(img_p), str(msk_p)))
        img_mask_pairs.extend(ct_pairs)
        print(f'Found {len(ct_pairs)} CT image-mask pairs from {ct_root}')

assert img_mask_pairs, (
    'No lung data found. Download one of:\n'
    '  - nikhilpandey360/chest-xray-masks-and-labels\n'
    '  - kmader/finding-lungs-in-ct-data\n'
    'from Kaggle and set the path manually.'
)

# Shuffle and cap
random.Random(CONFIG['SEED']).shuffle(img_mask_pairs)
img_mask_pairs = img_mask_pairs[: CONFIG['MAX_IMAGES']]
print(f'Using {len(img_mask_pairs)} total image-mask pairs')

In [ ]:
class LungSliceDataset(Dataset):
    '''Lung region segmentation from X-ray or CT images.'''

    def __init__(self, pairs, size=256, augment=False):
        self.size, self.augment = size, augment
        self.pairs = pairs  # list of (img_path, mask_path)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img_path, msk_path = self.pairs[i]
        x = np.array(Image.open(img_path).convert('L'), dtype=np.float32) / 255.0
        y = np.array(Image.open(msk_path).convert('L'), dtype=np.float32) / 255.0

        # Robust normalization → [0,1]
        lo, hi = np.percentile(x, (0.5, 99.5))
        span = max(hi - lo, 1e-6)
        x = (np.clip(x, lo, hi) - lo) / span

        if self.augment:
            if random.random() < 0.5:
                x, y = x[:, ::-1].copy(), y[:, ::-1].copy()
            k = random.randint(0, 3)
            if k:
                x, y = np.rot90(x, k).copy(), np.rot90(y, k).copy()
            x = x * np.random.uniform(0.92, 1.08)

        rs = (self.size, self.size)
        x = np.asarray(Image.fromarray((np.clip(x, 0, 1) * 255).astype(np.uint8), 'L').resize(rs, Image.BILINEAR), np.float32) / 255.0
        y = np.asarray(Image.fromarray((y * 255).astype(np.uint8), 'L').resize(rs, Image.NEAREST), np.float32)
        return torch.from_numpy(x[None]).float(), (y > 0.5).float()[None]

split = int(len(img_mask_pairs) * 0.85)
train_ds = LungSliceDataset(img_mask_pairs[:split], CONFIG['SIZE'], augment=True)
val_ds   = LungSliceDataset(img_mask_pairs[split:], CONFIG['SIZE'], augment=False)
train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH'], shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG['BATCH'], shuffle=False, num_workers=2, pin_memory=True)
print(f'train images {len(train_ds)} | val images {len(val_ds)}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for j in range(4):
    x, y = train_ds[random.randrange(len(train_ds))]
    axes[0, j].imshow(x[0], cmap='gray'); axes[0, j].axis('off')
    axes[1, j].imshow(x[0], cmap='gray'); axes[1, j].imshow(y[0], alpha=0.4, cmap='Reds'); axes[1, j].axis('off')
plt.suptitle('Image + lung mask overlay'); plt.tight_layout(); plt.show()

## 2 · Model, loss, metrics — shared MiniUNet

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.d1, self.d2, self.d3 = DoubleConv(1, base), DoubleConv(base, base*2), DoubleConv(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.u2, self.u1 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2), nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.c2, self.c1 = DoubleConv(base*4, base*2), DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.d1(x); s2 = self.d2(self.pool(s1)); b = self.d3(self.pool(s2))
        u = self.c2(torch.cat([self.u2(b), s2], dim=1))
        u = self.c1(torch.cat([self.u1(u), s1], dim=1))
        return self.out(u)

model = MiniUNet().to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters')

class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce, self.w, self.smooth = nn.BCEWithLogitsLoss(), dice_weight, smooth
    def forward(self, logits, target):
        bce = self.bce(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(2, 3))
        union = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()
        return bce + self.w * dice

@torch.no_grad()
def segmentation_metrics(logits, target, thr=0.5, eps=1e-7):
    pred = (torch.sigmoid(logits) > thr).float()
    tp = (pred * target).sum(); fp = (pred * (1 - target)).sum(); fn = ((1 - pred) * target).sum()
    return {
        'dice': float((2 * tp + eps) / (2 * tp + fp + fn + eps)),
        'iou': float((tp + eps) / (tp + fp + fn + eps)),
        'precision': float((tp + eps) / (tp + fp + eps)),
        'recall': float((tp + eps) / (tp + fn + eps)),
    }

criterion = DiceBCELoss(dice_weight=0.7)  # lean on Dice harder for mask quality
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LR'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

In [ ]:
def train_one_epoch(loader):
    model.train(); total = 0.0
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            loss = criterion(model(img), msk)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total += float(loss) * img.size(0)
    return total / max(1, len(loader.dataset))

@torch.no_grad()
def evaluate(loader):
    model.eval(); total = 0.0; agg = {'dice': [], 'iou': [], 'precision': [], 'recall': []}
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            logits = model(img)
        total += float(criterion(logits.float(), msk)) * img.size(0)
        for k, v in segmentation_metrics(logits.float(), msk).items():
            agg[k].append(v)
    out = {k: sum(v) / len(v) for k, v in agg.items()}
    out['loss'] = total / max(1, len(loader.dataset))
    return out

history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
best_dice, best_path = -1.0, OUT_DIR / 'lung_unet_best.pth'

for epoch in range(1, CONFIG['EPOCHS'] + 1):
    tl = train_one_epoch(train_loader)
    ev = evaluate(val_loader)
    scheduler.step(ev['dice'])
    history['train_loss'].append(tl); history['val_loss'].append(ev['loss'])
    history['val_dice'].append(ev['dice']); history['val_iou'].append(ev['iou'])
    star = ''
    if ev['dice'] > best_dice:
        best_dice = ev['dice']; torch.save(model.state_dict(), best_path); star = '  ← saved'
    print(f"epoch {epoch:02d} | train_loss {tl:.4f} | val_loss {ev['loss']:.4f} | "
          f"dice {ev['dice']:.4f} | iou {ev['iou']:.4f}{star}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('Dice+BCE loss'); ax[0].legend()
ax[1].plot(history['val_dice'], label='dice'); ax[1].plot(history['val_iou'], label='iou')
ax[1].set_title('validation overlap metrics'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
final = evaluate(val_loader)
print(json.dumps(final, indent=2))

onnx_path = OUT_DIR / 'lung_unet.onnx'
dummy = torch.randn(1, 1, CONFIG['SIZE'], CONFIG['SIZE'])
model.eval().cpu()
torch.onnx.export(
    model, dummy, str(onnx_path), opset_version=17,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch', 2: 'h', 3: 'w'}, 'logits': {0: 'batch', 2: 'h', 3: 'w'}},
)

import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
with torch.no_grad():
    ref = torch.sigmoid(model(dummy)).numpy()
raw = sess.run(None, {'input': dummy.numpy()})[0]
prob = raw if (raw.min() >= 0 and raw.max() <= 1) else 1 / (1 + np.exp(-raw))
diff = float(np.abs(prob - ref).max())
print(f'ONNX vs PyTorch max |Δ|: {diff:.2e}')
assert diff < 1e-3, 'export parity check failed'
print(f'Done → copy {onnx_path} to inference-service/models/lung_unet.onnx')

## Next steps & honest caveats

- These datasets provide lung region masks (not nodule/tumor masks).
  The model learns to delineate lung boundaries — useful for ROI extraction
  before a downstream nodule detector.
- For true nodule segmentation, consider LIDC-IDRI with multi-rater expert masks.
- Copy `outputs/lung_unet.onnx` → `inference-service/models/`, then set
  `MOCK_PREDICTIONS=0`.

> **This is a research prototype and not approved for clinical diagnosis.**